In [ ]:
pip install Anthropic google.generativeai openai mistralai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.3/509.3 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 14.0 MB/s eta 0:00:00


In [ ]:
#No need to run just to have to copy and paste below (doing in Markdown loses structure).
#Use press 'ctrl +/' all at once with all occupations highlighted to bulk add and remove hashs

# "13-2011.00",  # Accountants and Auditors
# "13-2061.00",  # Financial Examiners
# "15-1255.01",  # Video Game Designers
# "17-1022.00",  # Surveyors
# "19-1032.00",  # Foresters
# "19-2031.00",  # Chemists
# "19-3092.00",  # Geographers
# "21-1013.00",  # Marriage and Family Therapists
# "25-2021.00",  # Elementary School Teachers, Except Special Education
# "25-3041.00",  # Tutors
# "27-1022.00",  # Fashion Designers
# "27-3092.00",  # Court Reporters and Simultaneous Captioners
# "29-1021.00",  # Dentists, General
# "29-1211.00",  # Anesthesiologists
# "29-1224.00",  # Radiologists
# "29-1229.01",  # Allergists and Immunologists
# "31-1131.00",  # Nursing Assistants
# "31-9011.00",  # Massage Therapists
# "33-2011.00",  # Firefighters
# "33-3021.02",  # Police Identification and Records Officers
# "35-3023.00",  # Fast Food and Counter Workers
# "37-2011.00",  # Janitors and Cleaners, Except Maids and Housekeeping Cleaners
# "39-6012.00",  # Concierges
# "39-9011.00",  # Childcare Workers
# "39-9011.01",  # Nannies
# "41-2031.00",  # Retail Salespersons
# "41-9041.00",  # Telemarketers
# "43-3041.00",  # Gambling Cage Workers
# "43-5011.00",  # Cargo and Freight Agents
# "47-2111.00",  # Electricians
# "47-2181.00",  # Roofers
# "47-5011.00",  # Derrick Operators, Oil and Gas
# "47-5022.00",  # Excavating and Loading Machine and Dragline Operators, Surface Mining
# "49-3023.00",  # Automotive Service Technicians and Mechanics
# "51-3022.00",  # Meat, Poultry, and Fish Cutters and Trimmers
# "51-6011.00",  # Laundry and Dry-Cleaning Workers
# "51-9071.00",  # Jewelers and Precious Stone and Metal Workers
# "53-2011.00",  # Airline Pilots, Copilots, and Flight Engineers
# "53-3052.00",  # Bus Drivers, Transit and Intercity
# "53-7021.00",  # Crane and Tower Operators

In [39]:
# Import required libraries
import pandas as pd
import numpy as np
import json
import time
import os
from typing import Dict, List, Optional
from datetime import datetime
from anthropic import Anthropic
from openai import OpenAI
import google.generativeai as genai

# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
INPUT_JSON_FILE = "onetsoc_all_879_with_descriptions.json"
SCALES_FILE = "scales_long.txt"
PROMPT_FILE = "rating_instructions.txt"  # The new text file you created
DESTINATION_PATH = f"occupation_ratings_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

# METHOD 2: Specific SOC codes (Keep as you had it)
TEST_LIMIT = None
SPECIFIC_SOC_CODES = [
"51-9012.00",  # Separating, Filtering, Clarifying, Precipitating, and Still Machine Setters, Operators, and Tenders [801]
"51-9021.00",  # Crushing, Grinding, and Polishing Machine Setters, Operators, and Tenders [802]
"51-9022.00",  # Grinding and Polishing Workers, Hand [803]
"51-9023.00",  # Mixing and Blending Machine Setters, Operators, and Tenders [804]
"51-9031.00",  # Cutters and Trimmers, Hand [805]
"51-9032.00",  # Cutting and Slicing Machine Setters, Operators, and Tenders [806]
"51-9041.00",  # Extruding, Forming, Pressing, and Compacting Machine Setters, Operators, and Tenders [807]
"51-9051.00",  # Furnace, Kiln, Oven, Drier, and Kettle Operators and Tenders [808]
"51-9061.00",  # Inspectors, Testers, Sorters, Samplers, and Weighers [809]
"51-9071.00",  # Jewelers and Precious Stone and Metal Workers [810]
"51-9071.06",  # Gem and Diamond Workers [811]
"51-9081.00",  # Dental Laboratory Technicians [812]
"51-9082.00",  # Medical Appliance Technicians [813]
"51-9083.00",  # Ophthalmic Laboratory Technicians [814]
"51-9111.00",  # Packaging and Filling Machine Operators and Tenders [815]
"51-9123.00",  # Painting, Coating, and Decorating Workers [816]
"51-9124.00",  # Coating, Painting, and Spraying Machine Setters, Operators, and Tenders [817]
"51-9141.00",  # Semiconductor Processing Technicians [818]
"51-9151.00",  # Photographic Process Workers and Processing Machine Operators [819]
"51-9161.00",  # Computer Numerically Controlled Tool Operators [820]
"51-9162.00",  # Computer Numerically Controlled Tool Programmers [821]
"51-9191.00",  # Adhesive Bonding Machine Operators and Tenders [822]
"51-9192.00",  # Cleaning, Washing, and Metal Pickling Equipment Operators and Tenders [823]
"51-9193.00",  # Cooling and Freezing Equipment Operators and Tenders [824]
"51-9194.00",  # Etchers and Engravers [825]
"51-9195.00",  # Molders, Shapers, and Casters, Except Metal and Plastic [826]
"51-9195.03",  # Stone Cutters and Carvers, Manufacturing [827]
"51-9195.04",  # Glass Blowers, Molders, Benders, and Finishers [828]
"51-9195.05",  # Potters, Manufacturing [829]
"51-9196.00",  # Paper Goods Machine Setters, Operators, and Tenders [830]
"51-9197.00",  # Tire Builders [831]
"51-9198.00",  # Helpers--Production Workers [832]
"53-1041.00",  # Aircraft Cargo Handling Supervisors [833]
"53-1042.00",  # First-Line Supervisors of Helpers, Laborers, and Material Movers, Hand [834]
"53-1042.01",  # Recycling Coordinators [835]
"53-1043.00",  # First-Line Supervisors of Material-Moving Machine and Vehicle Operators [836]
"53-2011.00",  # Airline Pilots, Copilots, and Flight Engineers [837]
"53-2012.00",  # Commercial Pilots [838]
"53-2021.00",  # Air Traffic Controllers [839]
"53-2022.00",  # Airfield Operations Specialists [840]
"53-2031.00",  # Flight Attendants [841]
"53-3011.00",  # Ambulance Drivers and Attendants, Except Emergency Medical Technicians [842]
"53-3031.00",  # Driver/Sales Workers [843]
"53-3032.00",  # Heavy and Tractor-Trailer Truck Drivers [844]
"53-3033.00",  # Light Truck Drivers [845]
"53-3052.00",  # Bus Drivers, Transit and Intercity [846]
"53-4011.00",  # Locomotive Engineers [847]
"53-4013.00",  # Rail Yard Engineers, Dinkey Operators, and Hostlers [848]
"53-4022.00",  # Railroad Brake, Signal, and Switch Operators and Locomotive Firers [849]
"53-4031.00",  # Railroad Conductors and Yardmasters [850]
"53-4041.00",  # Subway and Streetcar Operators [851]
"53-5011.00",  # Sailors and Marine Oilers [852]
"53-5021.00",  # Captains, Mates, and Pilots of Water Vessels [853]
"53-5022.00",  # Motorboat Operators [854]
"53-5031.00",  # Ship Engineers [855]
"53-6011.00",  # Bridge and Lock Tenders [856]
"53-6021.00",  # Parking Attendants [857]
"53-6031.00",  # Automotive and Watercraft Service Attendants [858]
"53-6041.00",  # Traffic Technicians [859]
"53-6051.00",  # Transportation Inspectors [860]
"53-6051.01",  # Aviation Inspectors [861]
"53-6051.07",  # Transportation Vehicle, Equipment and Systems Inspectors, Except Aviation [862]
"53-6061.00",  # Passenger Attendants [863]
"53-7011.00",  # Conveyor Operators and Tenders [864]
"53-7021.00",  # Crane and Tower Operators [865]
"53-7031.00",  # Dredge Operators [866]
"53-7041.00",  # Hoist and Winch Operators [867]
"53-7051.00",  # Industrial Truck and Tractor Operators [868]
"53-7061.00",  # Cleaners of Vehicles and Equipment [869]
"53-7062.00",  # Laborers and Freight, Stock, and Material Movers, Hand [870]
"53-7062.04",  # Recycling and Reclamation Workers [871]
"53-7063.00",  # Machine Feeders and Offbearers [872]
"53-7064.00",  # Packers and Packagers, Hand [873]
"53-7065.00",  # Stockers and Order Fillers [874]
"53-7071.00",  # Gas Compressor and Gas Pumping Station Operators [875]
"53-7072.00",  # Pump Operators, Except Wellhead Pumpers [876]
"53-7073.00",  # Wellhead Pumpers [877]
"53-7081.00",  # Refuse and Recyclable Material Collectors [878]
"53-7121.00",  # Tank Car, Truck, and Ship Loaders [879]


]

# Initialize global variables for the model selector
ACTIVE_CLIENT = None
ACTIVE_PROVIDER = None # 'anthropic', 'openai', 'Mistral' or 'google'
ACTIVE_MODEL_NAME = None

def load_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

In [40]:
# ==========================================
# UNIFIED API CONFIGURATION (FILE-BASED)
# ==========================================
import os
import getpass
from anthropic import Anthropic
from openai import OpenAI
import google.generativeai as genai
from mistralai import Mistral  # Ensure you run: pip install mistralai

def load_api_key(filename, provider_name):
    """
    Attempts to read the API key from a local text file.
    Returns the key if found, otherwise returns None.
    """
    try:
        if os.path.exists(filename):
            with open(filename, 'r', encoding='utf-8') as f:
                key = f.read().strip()
                if key:
                    print(f"   Found API key in {filename}")
                    return key
        print(f"   ⚠️ Could not find or read {filename}")
        return None
    except Exception as e:
        print(f"   Error reading file: {e}")
        return None

# 1. Select Provider
print("Select which LLM provider to use:")
print("1. Anthropic (Claude 4.6 Opus)")
print("2. OpenAI (GPT-5.2)")
print("3. Google (Gemini 2.5 Pro)")
print("4. Mistral (Mistral 3 Large)")

choice = input("Enter number (1-4): ").strip()

# Initialize global variables
ACTIVE_CLIENT = None
ACTIVE_PROVIDER = None
ACTIVE_MODEL_NAME = None

# 2. Configure based on selection
if choice == '1':
    print("\n--- Selected: ANTHROPIC ---")
    api_key = load_api_key("ANTHROPIC_API_KEY.txt", "Anthropic")
    if not api_key:
        api_key = getpass.getpass("Enter your Anthropic API Key: ")

    ACTIVE_PROVIDER = "anthropic"
    ACTIVE_MODEL_NAME = "claude-opus-4-6"
    ACTIVE_CLIENT = Anthropic(api_key=api_key)
    print(f"✅ Ready: {ACTIVE_PROVIDER.upper()} ({ACTIVE_MODEL_NAME})")

elif choice == '2':
    print("\n--- Selected: OPENAI ---")
    api_key = load_api_key("OPENAI_API_KEY.txt", "OpenAI")
    if not api_key:
        api_key = getpass.getpass("Enter your OpenAI API Key: ")

    ACTIVE_PROVIDER = "openai"
    ACTIVE_MODEL_NAME = "gpt-5.2-2025-12-11"
    ACTIVE_CLIENT = OpenAI(api_key=api_key)
    print(f"✅ Ready: {ACTIVE_PROVIDER.upper()} ({ACTIVE_MODEL_NAME})")

elif choice == '3':
    print("\n--- Selected: GOOGLE ---")
    api_key = load_api_key("GOOGLE_API_KEY.txt", "Google")
    if not api_key:
        api_key = getpass.getpass("Enter your Google API Key: ")

    ACTIVE_PROVIDER = "google"
    ACTIVE_MODEL_NAME = "gemini-3-flash-preview"

    # --- CHANGE THIS LINE ---
    genai.configure(api_key=api_key, transport='rest')
    # ------------------------

    ACTIVE_CLIENT = genai.GenerativeModel(ACTIVE_MODEL_NAME)
    print(f"✅ Ready: {ACTIVE_PROVIDER.upper()} ({ACTIVE_MODEL_NAME})")

elif choice == '4':
    # --- MISTRAL ---
    print("\n--- Selected: MISTRAL ---")
    api_key = load_api_key("MISTRAL_API_KEY.txt", "Mistral")
    if not api_key:
        api_key = getpass.getpass("Enter your Mistral API Key: ")

    ACTIVE_PROVIDER = "mistral"
    # Mistral NeMo is their latest high-performance lightweight model
    ACTIVE_MODEL_NAME = "mistral-large-latest"
    ACTIVE_CLIENT = Mistral(api_key=api_key)
    print(f"✅ Ready: {ACTIVE_PROVIDER.upper()} ({ACTIVE_MODEL_NAME})")

else:
    print("❌ Invalid selection. Please run the cell again and choose 1, 2, 3, or 4.")

Select which LLM provider to use:
1. Anthropic (Claude 4.6 Opus)
2. OpenAI (GPT-5.2)
3. Google (Gemini 2.5 Pro)
4. Mistral (Mistral 3 Large)
Enter number (1-4): 2

--- Selected: OPENAI ---
   Found API key in OPENAI_API_KEY.txt
✅ Ready: OPENAI (gpt-5.2-2025-12-11)


In [11]:
def parse_scales_file(file_path: str) -> Dict:
    """
    Parse the scales text file and extract scale information.
    Returns a dictionary with scale names and their level descriptions.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    scales = {}
    current_scale = None
    current_scale_name = None

    lines = content.split('\n')

    for line in lines:
        line = line.strip()
        if line.startswith('Scale '):
            # Extract scale number and name
            parts = line.split(':')
            if len(parts) >= 2:
                scale_num = parts[0].replace('Scale ', '').strip()
                scale_name = parts[1].strip()
                current_scale_name = f"Scale {scale_num}: {scale_name}"
                scales[current_scale_name] = {}
        elif line.startswith('Level ') and ':' in line:
            # Extract level number and description
            parts = line.split(':', 1)
            level_num = parts[0].replace('Level ', '').strip()
            level_desc = parts[1].strip()
            if current_scale_name:
                scales[current_scale_name][f"Level {level_num}"] = level_desc

    return scales

In [12]:
def load_occupations_from_json(file_path: str, limit: Optional[int] = None, specific_codes: Optional[List[str]] = None) -> Dict[str, Dict]:
    """
    Load occupation data from JSON file.

    Args:
        file_path: Path to the JSON file
        limit: Optional limit for random sampling (e.g., process only first N occupations)
        specific_codes: Optional list of specific SOC codes to process

    Returns:
        Dictionary with occupation codes as keys and occupation data as values
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        all_occupations = json.load(f)

    print(f"Total occupations in JSON file: {len(all_occupations)}")

    # Method 1: Use specific SOC codes if provided
    if specific_codes:
        print(f"\nUsing SPECIFIC SOC CODES method")
        occupations = {}
        found_codes = []
        missing_codes = []

        for code in specific_codes:
            if code in all_occupations:
                occupations[code] = all_occupations[code]
                found_codes.append(code)
            else:
                missing_codes.append(code)

        print(f"Found {len(found_codes)} of {len(specific_codes)} requested SOC codes")

        if missing_codes:
            print(f"WARNING: The following SOC codes were not found in the JSON file:")
            for code in missing_codes:
                print(f"  - {code}")

        if not occupations:
            print("ERROR: None of the specified SOC codes were found in the JSON file")
            return {}

    # Method 2: Random sampling if limit is specified and no specific codes
    elif limit and not specific_codes:
        print(f"\nUsing RANDOM SAMPLING method")
        import random
        all_codes = list(all_occupations.keys())
        occupation_codes = random.sample(all_codes, min(limit, len(all_codes)))
        occupations = {code: all_occupations[code] for code in occupation_codes}
        print(f"Randomly selected {len(occupations)} occupations for testing")

    # Method 3: Process all occupations
    else:
        print(f"\nUsing PROCESS ALL method")
        occupations = all_occupations
        print(f"Processing all {len(occupations)} occupations")

    # Display loaded occupations
    print(f"\nOccupations to process ({len(occupations)} total):")
    for i, (code, data) in enumerate(occupations.items(), 1):
        print(f"  {i}. [{code}] {data['title']}")
        if i >= 10 and len(occupations) > 10:  # Show first 10 and indicate there are more
            print(f"  ... and {len(occupations) - 10} more")
            break

    return occupations

In [13]:
def create_rating_prompt(occupation_code: str, occupation_data: Dict, scales_info: Dict, prompt_template: str) -> str:
    """Populates the external prompt template with specific data."""

    occupation_title = occupation_data.get('title', 'Unknown')
    occupation_desc = occupation_data.get('description', 'No description available')
    task_list = occupation_data.get('tasks', [])
    domains_ranked = occupation_data.get('capability_domains_ranked', [])

    # Format scales
    scales_text = ""
    for scale_name, levels in scales_info.items():
        scales_text += f"\n{scale_name}\n"
        for level, desc in levels.items():
            scales_text += f"  {level}: {desc}\n"

    # Format tasks
    tasks_text = ""
    if task_list and isinstance(task_list, list) and len(task_list) > 0:
        tasks_text = "\nKEY TASKS FOR THIS OCCUPATION:\n"
        for i, task in enumerate(task_list[:15], 1):
            tasks_text += f"{i}. {task}\n"
    else:
        tasks_text = "\n(No specific task statements available for this occupation)\n"

    # Format capability domain rankings
    capability_domains_text = ""
    if domains_ranked and isinstance(domains_ranked, list) and len(domains_ranked) > 0:
        capability_domains_text = "\nCAPABILITY DOMAIN IMPORTANCE RANKING (highest to lowest):\n"
        for rank, domain in enumerate(domains_ranked, 1):
            domain_name = domain.get('domain', 'Unknown')
            score = domain.get('importance_score', 0)
            capability_domains_text += f"{rank}. {domain_name} (importance: {score:.4f})\n"
    else:
        capability_domains_text = "\n(No capability domain ranking data available for this occupation)\n"

    # Fill the template
    # Note: We use .format() but we must ensure the JSON braces in the prompt are escaped as {{ }}
    return prompt_template.format(
        occupation_code=occupation_code,
        occupation_title=occupation_title,
        occupation_desc=occupation_desc,
        tasks_text=tasks_text,
        scales_text=scales_text,
        capability_domains_text=capability_domains_text
    )

def query_llm(prompt: str) -> str:
    """Unified function to call the selected API."""
    if ACTIVE_PROVIDER == "anthropic":
        response = ACTIVE_CLIENT.messages.create(
            model=ACTIVE_MODEL_NAME,
            max_tokens=2000,
            temperature=0.2,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

    elif ACTIVE_PROVIDER == "openai":
        response = ACTIVE_CLIENT.chat.completions.create(
            model=ACTIVE_MODEL_NAME,
            temperature=0.2,
            messages=[{"role": "user", "content": prompt}],
            response_format={ "type": "json_object" }
        )
        return response.choices[0].message.content

    elif ACTIVE_PROVIDER == "google":
        # --- FIX: Enforce JSON mode via generation_config ---
        response = ACTIVE_CLIENT.generate_content(
            prompt,
            generation_config={"response_mime_type": "application/json"}
        )
        return response.text

    elif ACTIVE_PROVIDER == "mistral":
        # Mistral usage
        response = ACTIVE_CLIENT.chat.complete(
            model=ACTIVE_MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"} # Mistral also supports this now
        )
        return response.choices[0].message.content

    else:
        raise ValueError("No valid provider selected.")

def get_occupation_ratings(occupation_code: str, occupation_data: Dict, scales_info: Dict, prompt_template: str, max_retries: int = 3) -> Dict:
    """Gets ratings using the configured provider."""

    prompt = create_rating_prompt(occupation_code, occupation_data, scales_info, prompt_template)

    for attempt in range(max_retries):
        try:
            response_text = query_llm(prompt)

            # Clean JSON string (remove markdown fences if present)
            json_str = response_text
            if "```json" in response_text:
                json_start = response_text.find("```json") + 7
                json_end = response_text.find("```", json_start)
                json_str = response_text[json_start:json_end].strip()
            elif "{" in response_text:
                json_start = response_text.find("{")
                json_end = response_text.rfind("}") + 1
                json_str = response_text[json_start:json_end]

            return json.loads(json_str)

        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            if attempt == max_retries - 1:
                return {"ratings": {}, "error": str(e)}
            time.sleep(2)


In [41]:
def save_results_to_excel(results: List[Dict], output_file: str) -> pd.DataFrame:
    """
    Save the results to a well-formatted Excel file.
    """
    # Create a list of dictionaries for the main dataframe
    rows = []

    for result in results:
        row = {
            'O*NET-SOC Code': result['onet_soc_code'],
            'Occupation Title': result['occupation_title'],
            'Description': result['description'],
            'Number of Tasks': result['num_tasks'],
            'Processing Timestamp': result['timestamp']
        }

        # Add ratings for each scale
        for scale_name in result['ratings'].keys():
            rating_info = result['ratings'][scale_name]
            row[f'{scale_name} - Level'] = rating_info.get('level', -1)
            row[f'{scale_name} - Reasoning'] = rating_info.get('reasoning', '')
            row[f'{scale_name} - Level Description'] = rating_info.get('level_description', '')

        rows.append(row)

    # Create DataFrame
    df_results = pd.DataFrame(rows)

    # Save to Excel with formatting
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df_results.to_excel(writer, sheet_name='Occupation Ratings', index=False)

        # Get the workbook and worksheet
        workbook = writer.book
        worksheet = writer.sheets['Occupation Ratings']

        # Adjust column widths
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter

            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass

            adjusted_width = min(max_length + 2, 50)  # Cap at 50 characters
            worksheet.column_dimensions[column_letter].width = adjusted_width

    print(f"Results saved to {output_file}")
    return df_results

def save_results_to_json(results: List[Dict], output_file: str):
    """
    Save the results to a JSON file for programmatic access.
    """
    json_file = output_file.replace('.xlsx', '.json')
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"JSON results saved to {json_file}")

In [42]:
def process_and_save():
    if not ACTIVE_PROVIDER:
        print("❌ ERROR: No model selected. Please run one of the Configuration cells first.")
        return

    # 1. Load Data
    scales_data = parse_scales_file(SCALES_FILE)
    occupations = load_occupations_from_json(INPUT_JSON_FILE, specific_codes=SPECIFIC_SOC_CODES)
    prompt_template = load_text_file(PROMPT_FILE)

    results = []
    print(f"\n🚀 Starting processing with {ACTIVE_PROVIDER.upper()} ({ACTIVE_MODEL_NAME})...")

    # 2. Process
    for idx, (code, data) in enumerate(occupations.items(), 1):
        print(f"Processing {idx}/{len(occupations)}: {data.get('title')}")
        ratings = get_occupation_ratings(code, data, scales_data, prompt_template)

        results.append({
            'onet_soc_code': code,
            'occupation_title': data.get('title'),
            'description': data.get('description'),
            'num_tasks': len(data.get('tasks', [])),
            'ratings': ratings.get('ratings', {}),
            'model_used': ACTIVE_MODEL_NAME,
            'timestamp': datetime.now().isoformat()
        })
        time.sleep(1) # Rate limit safety

    # 3. Save Results (Flat File Protocol with Timestamp)
    # Generate timestamp for the filename
    now = datetime.now()
    timestamp_str = now.strftime('%Y%m%d_%H%M%S')

    # Define filename base: e.g., occupational_ratings_20240520_143005_gpt-4o
    filename_base = f"occupational_ratings_{timestamp_str}_{ACTIVE_MODEL_NAME}"

    # Ensure the destination directory exists (but no nested date folder)
    if not os.path.exists(DESTINATION_PATH):
        os.makedirs(DESTINATION_PATH, exist_ok=True)

    # Save JSON
    json_path = os.path.join(DESTINATION_PATH, f"{filename_base}.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2)

    # Save Excel
    excel_path = os.path.join(DESTINATION_PATH, f"{filename_base}.xlsx")
    save_results_to_excel(results, excel_path)

    print(f"\n✅ Done! Files saved directly to: {DESTINATION_PATH}")
    print(f"   - {os.path.basename(json_path)}")
    print(f"   - {os.path.basename(excel_path)}")

# Run the pipeline
process_and_save()

Total occupations in JSON file: 879

Using SPECIFIC SOC CODES method
Found 79 of 79 requested SOC codes

Occupations to process (79 total):
  1. [51-9012.00] Separating, Filtering, Clarifying, Precipitating, and Still Machine Setters, Operators, and Tenders
  2. [51-9021.00] Crushing, Grinding, and Polishing Machine Setters, Operators, and Tenders
  3. [51-9022.00] Grinding and Polishing Workers, Hand
  4. [51-9023.00] Mixing and Blending Machine Setters, Operators, and Tenders
  5. [51-9031.00] Cutters and Trimmers, Hand
  6. [51-9032.00] Cutting and Slicing Machine Setters, Operators, and Tenders
  7. [51-9041.00] Extruding, Forming, Pressing, and Compacting Machine Setters, Operators, and Tenders
  8. [51-9051.00] Furnace, Kiln, Oven, Drier, and Kettle Operators and Tenders
  9. [51-9061.00] Inspectors, Testers, Sorters, Samplers, and Weighers
  10. [51-9071.00] Jewelers and Precious Stone and Metal Workers
  ... and 69 more

🚀 Starting processing with OPENAI (gpt-5.2-2025-12-11)...